# Extract Attention Matrices and Analyze Model Metrics

This notebook demonstrates how to load the Gemma 3 model, track its memory footprint, and analyze its attention mechanisms using advanced metrics (entropy, distance, sparsity).

In [8]:
# Setup and Imports
import os

import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

from attention_utils import (
    MemoryTracker,
    analyze_prompt,
    print_memory_usage,
    print_model_memory_footprint,
)
from example_prompts import ALL_PROMPTS, RAG_PROMPTS, SIMPLE_PROMPTS

load_dotenv()
torch.set_grad_enabled(False)

if os.getenv("HF_TOKEN"):
    login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
# 1. Initial Memory Status
print_memory_usage("Baseline")


[Baseline] Memory Usage Report
Process Memory:
  RSS (Resident Set Size): 185.28 MB
  VMS (Virtual Memory):    426063.30 MB

System Memory:
  Total:     16384.00 MB
  Available: 4748.17 MB
  Used:      71.0%

MPS Memory:
  Allocated: 1936.31 MB
  Driver Allocated: 3046.69 MB



In [ ]:
# 2. Load Model on Device
checkpoint = "google/gemma-3-1b-it"
device = torch.device(
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)

with MemoryTracker(f"Loading {checkpoint}"):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForCausalLM.from_pretrained(
        checkpoint,
        device_map="auto",
        dtype=torch.float16 if device.type in ["cuda", "mps"] else torch.float32,
        attn_implementation="eager",
    )
    model.eval()

print_model_memory_footprint(model, checkpoint)

`torch_dtype` is deprecated! Use `dtype` instead!



──────────────────────────────────────────────────
📊 Memory Tracking: Loading google/gemma-3-1b-it
──────────────────────────────────────────────────
  ⏱️  Duration:    4.42s
  💾 RSS Change:  +156.56 MB
  🍎 MPS Change:  +178.26 MB
──────────────────────────────────────────────────


════════════════════════════════════════════════════════════
🧠 google/gemma-3-1b-it
════════════════════════════════════════════════════════════

📦 Memory Footprint:
    Parameters:        1907.13 MB
    Buffers:           0.00 MB
    Total:             1907.13 MB
    Num Parameters:    999,885,952
    Trainable Params:  999,885,952

🏗️  Architecture:
    Layers:            26
    Heads (Q/KV):      4 / 1
    Head Dim:          288
════════════════════════════════════════════════════════════



## Simple Prompts Analysis
Quick tests with short prompts to see basic attention patterns.

In [ ]:
# Analyze simple prompts
for key, text in SIMPLE_PROMPTS.items():
    analyze_prompt(key=key, prompt=text, model=model, tokenizer=tokenizer, device=device)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
🔬 Analyzing Prompt: [simple_fact]
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

──────────────────────────────────────────────────
📊 Memory Tracking: Inference - simple_fact
──────────────────────────────────────────────────
  ⏱️  Duration:    5.64s
  💾 RSS Change:  +430.20 MB
  🍎 MPS Change:  +16.38 MB
──────────────────────────────────────────────────


💬 Model Answer:
   
```python
def find_capital(city):
    """
    This function takes a city name as input and returns the capital of that city.
    """
    if city == "Paris":
        return "Paris"
    else:
        return "Unknown"
```

**Explanation:**

1.  **`def find_capital(city):`**: This line defines a function named `find_capital` that takes one argument, `city`.


📐 Sequence Info:
    Layers: 26 | Heads: 4 | Tokens: 115
    KV Cache: 3.2849 MB

📊 Attention Stats (Last Layer, Avg):
    Entropy:      2.6232 (Norm: 0.55)
    Avg Distance: 23.74 token


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
🔬 Analyzing Prompt: [reasoning]
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


## RAG Agent Context Analysis

This section tests attention patterns with a realistic **RAG (Retrieval-Augmented Generation)** context.
The context includes:
- System instructions
- 6 retrieved documents (~1500 tokens)
- A user question

This simulates a typical agentic workload where the model must:
1. Attend to relevant documents
2. Identify key information
3. Generate a grounded response

In [ ]:
# Preview the RAG context size
sample_rag = RAG_PROMPTS["rag_founder"]
tokens = tokenizer.encode(sample_rag)
print(f"RAG Context Token Count: {len(tokens)}")
print(f"RAG Context Character Count: {len(sample_rag)}")

In [ ]:
# Analyze a single RAG prompt in detail
analyze_prompt(
    key="rag_founder",
    prompt=RAG_PROMPTS["rag_founder"],
    model=model,
    tokenizer=tokenizer,
    device=device
)

In [ ]:
# Compare memory and attention for different RAG questions
# These all use the same context but ask different questions
for key in ["rag_pricing", "rag_tech_stack"]:
    analyze_prompt(
        key=key,
        prompt=RAG_PROMPTS[key],
        model=model,
        tokenizer=tokenizer,
        device=device
    )

## Final Memory State

In [ ]:
print_memory_usage("Final State")